# 01 — QC, revue et splits groupés (tâches 06 à 08)


Le fichier de revue est écrit avant le contrôle bloquant. Les exclusions validées alimentent directement le split.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import experiment_config as cfg
from src.io.database_h5 import load_nir_uco_h5
from src.workflows.protocol_split import (
    build_protocol_manifest,
    build_split_diagnostics,
)
from src.workflows.quality_check import (
    add_robust_spectral_qc,
    apply_qc_reviews,
    build_image_qc_table,
    build_pixel_spectral_qc_table,
    build_image_qc_warnings,
    build_object_qc_table,
    build_object_qc_warnings,
    build_object_shape_check_tables,
    build_qc_alerts_table,
    build_qc_exclusion_report,
    build_qc_protocol,
    build_qc_visual_review_report,
    check_missing_required_fields,
    merge_existing_reviews_or_initialize,
    validate_qc_review_closure,
)
from src.utils import save_parquet
from src.visualization.plot_objects import plot_object_area_distribution
from src.visualization.plot_images import plot_image_overlay

H5_PATH = PROJECT_ROOT.joinpath(*cfg.DATABASE_H5_RELATIVE_PATH)
RESULTS_DIR = PROJECT_ROOT.joinpath(*cfg.QC_RESULTS_RELATIVE_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = {
    key: RESULTS_DIR / filename for key, filename in cfg.QC_OUTPUT_FILENAMES.items()
}
VERSIONED_REVIEW = PROJECT_ROOT.joinpath(*cfg.QC_REVIEW_DECISIONS_RELATIVE_PATH)
object_db, image_db = load_nir_uco_h5(
    H5_PATH, reconstruct_heavy_object_arrays=True
)


In [2]:
from src.protocol_governance import (
    build_protocol_configuration,
    sha256_payload,
    verify_frozen_protocol,
)
PROTOCOL_DIR = PROJECT_ROOT.joinpath(
    *cfg.PROTOCOL_ARTIFACT_RELATIVE_DIR
)

protocol_verification = verify_frozen_protocol(
    PROTOCOL_DIR,
    strict=True,
)

display(protocol_verification)

,check,passed,detail
0,all_frozen_artifacts_exist,True,missing=[]
1,configuration_sha256_matches_current_protocol,True,expected=9e87eac5b065a9fae9ef1ff543981234bfbda...
2,inference_plan_sha256_matches_current_protocol,True,expected=10624351f77a2b18a37b8a51b766be759e4cc...
3,planned_contrasts_sha256_matches_current_protocol,True,expected=5a557c1c13441366f7795cbed2504f70516af...
4,checks_file_checksum_matches_lock,True,expected=9e972be2cfa8cfe634a304b706c12b9cab103...
5,inference_plan_file_checksum_matches_lock,True,expected=3b34fa6331ce14de3bb2b63202cbb821333cc...
6,manifest_file_checksum_matches_lock,True,expected=80fd7411e9a0493d37f60ec59c229702ee3b4...
7,planned_contrasts_file_checksum_matches_lock,True,expected=14abc9529b2fa764bc9412c0122e5c981c55a...
8,lock_checksum_is_valid,True,expected=5d66e659d7da4e69fa647123058bcea08d33f...


In [3]:
plot_object_area_distribution(
    object_db,
    kind="box",
    facet_by_batch=True,
    points="outliers",
)

In [4]:
# ---------------------------------------------------------------------------
# Pixel-level spectral QC
# ---------------------------------------------------------------------------

pixel_spectral_qc = build_pixel_spectral_qc_table(
    object_db,
    policy=cfg.SPECTRAL_PIXEL_VALIDITY_POLICY,
)
save_parquet(pixel_spectral_qc, OUTPUT["pixel_spectral_qc"])

pixel_exclusions = pixel_spectral_qc.loc[
    ~pixel_spectral_qc["analysis_valid"]
].copy()
save_parquet(pixel_exclusions, OUTPUT["pixel_exclusions"])

print("Pixel spectral QC")
print("-----------------")
print("Total pixels:", len(pixel_spectral_qc))
print(
    "Valid:",
    int(pixel_spectral_qc["analysis_valid"].sum()),
)
print(
    "Invalid:",
    int((~pixel_spectral_qc["analysis_valid"]).sum()),
)

display(
    pixel_spectral_qc["invalid_reason"]
    .value_counts(dropna=False)
    .rename_axis("reason")
    .reset_index(name="n_pixels")
)

display(
    pixel_exclusions[
        [
            "object_id",
            "source_image",
            "row",
            "col",
            "n_zero",
            "n_bands",
            "min_reflectance",
            "invalid_reason",
        ]
    ]
)

Pixel spectral QC
-----------------
Total pixels: 104183
Valid: 104077
Invalid: 106


,reason,n_pixels
0,valid,104077
1,all_zero_spectrum,106


,object_id,source_image,row,col,n_zero,n_bands,min_reflectance,invalid_reason
654,alm1pea1_obj008,alm1pea1,120,137,61,61,0.0,all_zero_spectrum
1184,alm1pea1_obj015,alm1pea1,150,89,61,61,0.0,all_zero_spectrum
1913,alm1pea1_obj023,alm1pea1,194,70,61,61,0.0,all_zero_spectrum
2484,alm1pea1_obj029,alm1pea1,228,200,61,61,0.0,all_zero_spectrum
3367,alm1pea1_obj038,alm1pea1,278,52,61,61,0.0,all_zero_spectrum
...,...,...,...,...,...,...,...,...
98368,peanut3_obj010,peanut3,119,237,61,61,0.0,all_zero_spectrum
98453,peanut3_obj011,peanut3,122,61,61,61,0.0,all_zero_spectrum
98950,peanut3_obj019,peanut3,163,142,61,61,0.0,all_zero_spectrum
99359,peanut3_obj026,peanut3,189,127,61,61,0.0,all_zero_spectrum


In [5]:
image_qc = build_image_qc_table(image_db)
object_qc = build_object_qc_table(
    object_db,
    image_db=image_db,
    pixel_qc_df=pixel_spectral_qc,
    border_margin=cfg.QC_BORDER_MARGIN,
)
object_qc = add_robust_spectral_qc(object_qc, object_db, pixel_validity_policy=cfg.SPECTRAL_PIXEL_VALIDITY_POLICY)
image_warnings = build_image_qc_warnings(image_qc)
object_warnings = build_object_qc_warnings(object_qc)
missing_fields = check_missing_required_fields(image_db, object_db)
_, bad_shapes = build_object_shape_check_tables(object_db, image_db)
alerts = build_qc_alerts_table(
    image_warnings,
    object_warnings,
    missing_fields,
    bad_shapes,
)
save_parquet(image_qc, OUTPUT["image_summary"])
save_parquet(object_qc, OUTPUT["object_summary"])
save_parquet(alerts, OUTPUT["alerts"])

# All cube-dependent checks are complete. Reconstructed crops retain
# references to the full image cubes and can otherwise keep several GB
# alive while the PDF report is rendered.
for image in image_db.values():
    image.pop("cube", None)
for obj in object_db.values():
    obj.pop("cube_crop", None)
    obj.pop("mask_global", None)
    obj.pop("image_ref_crop", None)

import gc
gc.collect()


0

In [6]:
print("Versioned review source:", VERSIONED_REVIEW)
print("Exists:", VERSIONED_REVIEW.exists())

if not VERSIONED_REVIEW.exists():
    raise FileNotFoundError(
        "Versioned QC review decisions are missing. "
        f"Expected: {VERSIONED_REVIEW}"
    )

review = merge_existing_reviews_or_initialize(
    qc_alerts_df=alerts,
    review_source=VERSIONED_REVIEW,
)

save_parquet(
    review,
    OUTPUT["review"],
)

build_qc_visual_review_report(
    alerts,
    object_db,
    image_db,
    object_qc,
    OUTPUT["visual_review"],
)

review

Versioned review source: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\docs\protocol\8tracks_v4\qc_review_decisions.parquet
Exists: True


,record_type,record_id,flag_type,review_status,review_decision,reviewer,review_date,review_comment,review_evidence
0,object,alm1pea1_obj030,possible_merged_object,reviewed,accept_as_is,visual_review,2026-08-07,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
1,object,alm3pea3_obj005,possible_merged_object,reviewed,accept_as_is,visual_review,2026-08-07,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
2,object,alm3pea3_obj029,possible_merged_object,reviewed,accept_as_is,visual_review,2026-08-07,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
3,object,pea3_pos3_obj004,possible_merged_object,reviewed,accept_as_is,visual_review,2026-08-07,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
4,object,pea3_pos3_obj008,possible_merged_object,reviewed,accept_as_is,visual_review,2026-08-07,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
5,object,almond2_obj032,possible_merged_object,reviewed,accept_as_is,visual_review,2026-08-07,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
6,object,alm3pea2_obj026,robust_spectral_outlier,reviewed,accept_as_is,visual_review,2026-08-07,Ignoring for now,qc_visual_review_report.pdf
7,object,alm3pea4_obj008,robust_spectral_outlier,reviewed,accept_as_is,visual_review,2026-08-07,Ignoring for now,qc_visual_review_report.pdf


In [7]:
# Point de contrôle humain: le fichier pending et le PDF existent déjà.
validate_qc_review_closure(review)
resolved_alerts = apply_qc_reviews(alerts, review, require_complete=True)

In [8]:
exclusions = build_qc_exclusion_report(
    resolved_alerts
)

qc_protocol = build_qc_protocol(
    alerts,
    review,
    exclusions,
    pixel_exclusion_manifest=pixel_exclusions,
    spectral_pixel_policy=(
        cfg.SPECTRAL_PIXEL_VALIDITY_POLICY
    ),
)

split_manifest, split_checks = (
    build_protocol_manifest(
        image_db,
        object_db,
        exclusion_manifest=exclusions,
        strict=True,
    )
)

split_diagnostics = build_split_diagnostics(
    split_manifest,
    object_db,
)

save_parquet(
    exclusions,
    OUTPUT["exclusion_manifest"],
)

save_parquet(
    qc_protocol,
    OUTPUT["protocol"],
)

save_parquet(
    split_manifest,
    OUTPUT["split_manifest"],
)

save_parquet(
    split_diagnostics,
    OUTPUT["split_diagnostics"],
)

qc_protocol

,protocol_version,qc_policy_hash,spectral_pixel_policy_hash,pixel_exclusion_hash,alerts_hash,review_hash,n_alerts,n_pending,n_excluded,n_pixel_excluded,closure_status
0,8tracks_v5,978faa83e2349368c03e950de7ea938837c6d0912584b4...,95359d91fdb0d56f38441b43f9ab60bd6873cb5b318028...,7da582b59127b93bec2fdb9da586047d6b5a1adecd6912...,f88b7e836a7cab03030503877e406801735576586f9363...,d7ccf06d0ed45eb45f3728f5182e6d944e650b6aa0a104...,8,0,0,106,closed


In [9]:
assert int(
    qc_protocol.iloc[0]["n_pixel_excluded"]
) == len(pixel_exclusions)